<a href="https://colab.research.google.com/github/ironspiritjeff/Ames-Iowa-House-Price-Regression-Model/blob/main/House_Price_Regression_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏡 End-to-End House Price Predictor - Regression Model

**Dataset:** House Prices - Advanced Regression Techniques
  * https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/overview

**Objective:** Predict the continuous target variable (Price) using Scikit-Learn, Pandas, and NumPy.  
**Methodology:** Use Just-In-Time (JIT) Documentation Retrieval & Deep Problem Solving to solve each section before writing code. Do not use video tutorials.

**Reference Notebook Tooling:** Use only to verify architectural direction, never to copy syntax.

---

## 🧭 The JIT Retention Rules
1. **The 5-Minute Sandbox:** When stuck or hitting an error, spend 5 minutes reading the official documentation or the Python error trace before looking at the Kaggle reference notebook.
2. **The "Why" Test:** Before running any code cell, ask yourself: *What exactly am I expecting this line to return, and why am I using this specific library over another?*
3. **No Copy-Pasting:** Every line of code must be manually typed out by your own fingers to build syntax familiarity.

---

## 🏃‍♂️ Phase 1: Data Isolation & Leakage Prevention
*Goal: Secure your evaluation data before your brain or your code can look at it.*

### ❓ Core Question: How do I split data safely to avoid Data Leakage?
*   **The Theory:** Data leakage happens when training features accidentally look at the validation data (like calculating the mean of the *entire* dataset before splitting). This makes your model look smart during testing but fail completely in production.
*   *JIT Reading Anchor:* Read the Scikit-Learn user guide on "Data Leakage" to understand why we split data *before* preprocessing.

### 🛠️ Execution Checklist
- [ ] **Step 1.1:** Load the `train.csv` file using Pandas. Inspect the dimensions using `.shape`.
- [ ] **Step 1.2:** Separate the target variable (`SalePrice`) into a vector `y` and the independent features into a DataFrame `X`.
- [ ] **Step 1.3:** Use `sklearn.model_selection.train_test_split` to create `X_train`, `X_val`, `y_train`, and `y_val` (use an 80/20 split).
- [ ] **Step 1.4:** Verify that your future preprocessors are only ever `.fit()` on `X_train` and merely `.transform()` on `X_val`.

---


## 📊 Phase 2: Target Variable Analysis & Normalization
*Goal: Analyze and stabilize the mathematical distribution of your predictions.*

### ❓ Core Question: How do I check if my target variable is "Skewed"?
*   **The Theory:** If your housing prices are heavily skewed (e.g., a few massive $3,000,000 mansions dragging out a long right tail), standard regression algorithms struggle to capture the pattern accurately. Transforming the target distribution into a normal bell curve allows algorithms to converge efficiently.
*   *JIT Reading Anchor:* Read the NumPy documentation for `log1p` and `expm1` to understand how to reverse this calculation later.

### 🛠️ Execution Checklist
- [ ] **Step 2.1:** Use Matplotlib or Seaborn to plot a raw histogram of `y_train`.
- [ ] **Step 2.2:** Calculate the skewness statistic using the Pandas feature `.skew()`.
- [ ] **Step 2.3:** If the skewness score is greater than 1.0, apply a logarithmic transformation to both `y_train` and `y_val` using `numpy.log1p()`.
- [ ] **Step 2.4:** Re-plot the histogram of your log-transformed target to verify it resembles a standard bell curve.

---

## 🧹 Phase 3: Feature Discrimination & Advanced Preprocessing
*Goal: Deal with the 79 columns by breaking them down into logical, machine-readable types.*

### ❓ Core Question A: How do I handle missing data without dropping rows?
*   **The Theory:** Real datasets have gaps. Indiscriminately dropping rows removes valuable training samples. Replacing missing values mathematically requires an intentional strategy: numeric features can be filled with middle values (like the median) to mitigate the influence of outliers, while categorical gaps require a placeholder label or the most frequent class.
*   **JIT Reading Anchor:** Look up the parameters for `sklearn.impute.SimpleImputer`.

### ❓ Core Question B: How do I turn text categories into numbers properly?
*   **The Theory:** Algorithms only process math, not strings. Unordered columns (Nominal text like `Neighborhood`) must be spread across binary columns using One-Hot Encoding so the model doesn't assume neighborhood "B" is mathematically greater than neighborhood "A". Ordered columns (Ordinal text like `ExterQual`: Poor, Fair, Typical, Good, Excellent) must be mapped to ranked integers.
*   **JIT Reading Anchor:** Review `sklearn.preprocessing.OneHotEncoder` and `sklearn.preprocessing.OrdinalEncoder`.

### 🛠️ Execution Checklist
- [ ] **Step 3.1:** Identify your missing data profile using `X_train.isna().sum()`. Note which columns are missing values.
- [ ] **Step 3.2:** Programmatically split your columns into a `numeric_features` list and a `categorical_features` list.
*   [ ] **Step 3.3:** Use `sklearn.impute.SimpleImputer` to fill missing numeric data with the median.
*   [ ] **Step 3.4:** Identify which categorical columns are **Ordinal** (e.g., `ExterQual`: Ex, Gd, TA, Fa) vs. **Nominal** (e.g., `Neighborhood`).
- [ ] **Step 3.5:** Apply `OneHotEncoder(handle_unknown='ignore', sparse_output=False)` to categorical text columns to prevent dimension mismatches when checking validation data.

*   *JIT Reading Anchor:* Look up the documentation parameters for `OneHotEncoder(handle_unknown='ignore', sparse_output=False)` to prevent dimension errors during validation.
---

## 🤖 Phase 4: Baseline Training & Evaluation Framework
*Goal: Build a solid evaluation metric to track your model's true performance.*

### ❓ Core Question: How do I evaluate a regression model?
*   **The Theory:** Classification metrics like accuracy scores do not work when predicting continuous prices. If a house costs $300,000 and your model guesses $295,000, it is technically "wrong," but practically excellent. You must measure real dollar errors using Mean Absolute Error (MAE) for average distance, and Root Mean Squared Error (RMSE) to penalize rare, massive misses.
*   **JIT Reading Anchor:** Read the Scikit-Learn metrics user guide covering `mean_absolute_error` and `root_mean_squared_error`.

### 🛠️ Execution Checklist
- [ ] **Step 4.1:** Write a reusable Python function named `evaluate_regression(y_true, y_pred)` that explicitly prints out calculated MAE, RMSE, and $R^2$ scores.
- [ ] **Step 4.2:** Instantiate a baseline `sklearn.ensemble.RandomForestRegressor`.
- [ ] **Step 4.3:** Fit the regressor on your processed training arrays and output predictions from your validation feature set (`X_val`).
- [ ] **Step 4.4:** Convert your validation predictions back to real dollar limits using `numpy.expm1()` before running them through your evaluation function.

*   *JIT Reading Anchor:* Read the Scikit-Learn documentation metrics module specifically comparing MAE (average dollar error) to RMSE (penalty for large misses).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn
from sklearn.model_selection import train_test_split
import random

Import and view data:

This is the actual dataset, showing the first 10 samples of 1460 houses.

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Ames Housing Dataset/train.csv')
print(f"Shape of dataframe: {df.shape}")

In [ ]:
df.head(10)

There are 81 Columns, including the "SalePrice" columm. Because "SalePrice" is what the model will learn to predict, it needs to be separated from the other 80 columns.

Since the "Id" column has no predictive value it will also be removed, leaving 79 columns of actual house features.

From now on, the "SalePrice" column will be referred to as "y", and the rest of the data will be called "X".

The goal is to build a model that uses the features of a house to predict a sale price. In other words, to build a model that uses "X" to predict "y".

In [ ]:
X = df.drop(["SalePrice", "Id"], axis = 1)
y = df["SalePrice"]
print(f"First 5 items in target column, now called 'y':\n{y.head()}")



Split the whole dataset before making changes to it.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=7)

In [ ]:
print(f"X_train shape: {X_train.shape}    (These feature columns will be used for training)")
print(f"X_val shape: {X_val.shape}       (These feature columns will be used for validation)")
print(f"y_train shape: {y_train.shape}       (These target columns will be used for training)")
print(f"y_val shape: {y_val.shape}          (These target columns will be used for validation)")

## 📊 Phase 2: Target Variable Analysis & Normalization
*Goal: Analyze and stabilize the mathematical distribution of your predictions.*


In [ ]:
custom_bins = [0,100000,200000,300000,400000,500000,600000,700000,800000,900000]

plt.figure(figsize=(10, 5))
counts, bins, _ = plt.hist(y_train,
                           bins=custom_bins,
                           color='skyblue', edgecolor='black')

plt.xlabel('Sale Price')
plt.ylabel('Frequency')
plt.title("SalePrice Histogram")
plt.show()

print("Bin counts:", counts)
print("Bin edges:", bins)

In [ ]:
skew = y_train.skew()
skew

###Problem: Data is skewed (ie, the histogram above has a "tail" because of a few very expensive houses)
The SalePrice Histogram (and the skew value of ~1.85) show that the data is highly skewed, meaning that a few extreme outliers will cause the model to learn the wrong pattern.

###Solution: Apply `log1p` transformation to reduce the effect of these extreme outliers.
To fix this, a logarithmic transformation will compress the value of the outliers. Specifically, `log1p` will be applied, which reduces the impact of large values, bringing the data closer to a symetrical normal-like distribution, which is optimal for many machine learning algorithms.

**Note:** This will be reversed later so that once the model has made predictions, those predictions are brought back to their original scale.

In [ ]:
# log1p will be applied to the all target values, in both the training and validation splits.
log_y_train = np.log1p(y_train)
log_y_val = np.log1p(y_val)

In [ ]:
print(log_y_train.skew())
print(log_y_val.skew())

The new skew values are much closer to zero, which is what we want. (Between 0-0.5 is  normally recommended.) Let's visualize this with a new histogram which shows the log transformed Sale Prices, for both training and validation.

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.hist(log_y_train, bins=10, color='darkblue', edgecolor='lightblue')
plt.xlabel("Training Sale Price")
plt.ylabel("Sale Frequency")
plt.title("Log transformed prices for training")

plt.subplot(1,2,2)
plt.hist(log_y_val, bins=10, color='darkblue', edgecolor='lightblue')
plt.xlabel("Validation Sale Price")
plt.ylabel("Sale Frequency")
plt.title("Log transformed prices for validation")

plt.tight_layout()
plt.show()

The Sale Prices are now compressed in to more of a bell shaped curve, representing all of the data, but with no extreme outliers.

---

## 🧹 Phase 3: Feature Discrimination & Advanced Preprocessing
*Goal: Deal with the 79 columns by breaking them down into logical, machine-readable types.*

### ❓ Core Question A: How do I handle missing data without dropping rows?
*   **The Theory:** Real datasets have gaps. Indiscriminately dropping rows removes valuable training samples. Replacing missing values mathematically requires an intentional strategy: numeric features can be filled with middle values (like the median) to mitigate the influence of outliers, while categorical gaps require a placeholder label or the most frequent class.
*   **JIT Reading Anchor:** Look up the parameters for `sklearn.impute.SimpleImputer`.

### ❓ Core Question B: How do I turn text categories into numbers properly?
*   **The Theory:** Algorithms only process math, not strings. Unordered columns (Nominal text like `Neighborhood`) must be spread across binary columns using One-Hot Encoding so the model doesn't assume neighborhood "B" is mathematically greater than neighborhood "A". Ordered columns (Ordinal text like `ExterQual`: Poor, Fair, Typical, Good, Excellent) must be mapped to ranked integers.
*   **JIT Reading Anchor:** Review `sklearn.preprocessing.OneHotEncoder` and `sklearn.preprocessing.OrdinalEncoder`.

### 🛠️ Execution Checklist
- [ ] **Step 3.1:** Identify your missing data profile using `X_train.isna().sum()`. Note which columns are missing values.
- [ ] **Step 3.2:** Programmatically split your columns into a `numeric_features` list and a `categorical_features` list.
*   [ ] **Step 3.3:** Use `sklearn.impute.SimpleImputer` to fill missing numeric data with the median.
*   [ ] **Step 3.4:** Identify which categorical columns are **Ordinal** (e.g., `ExterQual`: Ex, Gd, TA, Fa) vs. **Nominal** (e.g., `Neighborhood`).
- [ ] **Step 3.5:** Apply `OneHotEncoder(handle_unknown='ignore', sparse_output=False)` to your remaining nominal text columns to prevent dimension mismatches when checking validation data.

*   *JIT Reading Anchor:* Look up the documentation parameters for `OneHotEncoder(handle_unknown='ignore', sparse_output=False)` to prevent dimension errors during validation.
---

Now we get into the nitty gritty. Machine learning models can only understand numbers. They don't know what to do with missing values, or with English words we use (called "strings"). The problem is, the Ames Housing Prices dataset, like most real-world datasets, is missing values and it also has non-numeric values too.

We'll tackle these problems one at a time, starting with missing values.

Note: In the actual dataset shown above, these are shown as 'NaN', meaning 'Not a Number'.

Let's start by visualizing how incomplete the dataset is, to deal with the problem of missing values first. We'll do this by counting how many values are absent each column.



In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
print(X_train.isna().sum())

Just by looking at the missing values shown above, we can see that around 1/3 of all columns contain some (or many!) missing values. This means that when the model is trained on this dataset, it will either crash, or it will substitute a value of `0` in these columns, when in reality the number is unknown. Unless this is fixed, the model will either not perform at all, or it will learn from false patterns.

There are two types of values:
* Some of the columns have values that are meant to be expressed as numbers. For example, YearBuilt: "1976". These values are called numeric features.

* The rest of the columns have values that are expressed in strings of text. For example, GarageType: "Attchd", and "Detchd". These values are called categorical features.

It's best to handle these two types of features using different methods to fill them.
* For numeric features, the median will be substituted. This is a common approach to fill missing numbers with the middle number (not the average).
* For categorical features, a "Missing" category can be created. The model will learn whether the fact that a value is missing in a particular column has any predictive power for the sale price.


Later on, categorical features will also need to be converted into numbers which the model can meaningfully use.

Lets start by taking a look at how many columns contain numeric features, and how many contain categorical features.

### Step 3.2: Programmatically split columns into numeric and categorical lists

We'll use pandas `select_dtypes` to automatically identify columns that are numeric and those that are considered categorical (object type in pandas).

In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include='object').columns.tolist()

print(f"Numeric features ({len(numeric_features)}):\n {numeric_features}\n")
print(f"Categorical features ({len(categorical_features)}):\n{categorical_features}")

###Step 3.3: Use sklearn.impute.SimpleImputer to fill missing numeric data with the median.

Note: it is absolutely crucial to use `fit.transform` below only with training data, not validation data. This is because `fit.transform` learns the patterns of the exact data it's directly applied to. When we fill the missing values of the validation set, we don't want those values to be based on anything learned in the validation set. That would be like cheating on the exam!

Instead, we want missing values in the validation set to be filled the same way the model has learned to do during training, from the training set.

`fit.transform` will learn the patterns from the training data, and `.transform` will apply those previously learned patterns to the validation data.

In [ ]:
from sklearn.impute import SimpleImputer

num_imputer = SimpleImputer(missing_values=np.nan, strategy="median")
cat_imputer = SimpleImputer(missing_values=np.nan, strategy="most_frequent")
         #Alternative optional strategy for cat_imputer: "constant", fill_value="Missing")

X_train_num_imputed = num_imputer.fit_transform(X_train[numeric_features])
X_train_cat_imputed = cat_imputer.fit_transform(X_train[categorical_features])

X_val_num_imputed = num_imputer.transform(X_val[numeric_features])
X_val_cat_imputed = cat_imputer.transform(X_val[categorical_features])

print(f"Imputed Training Numeric Shape: {X_train_num_imputed.shape}")
print(f"Imputed Training Categorical Shape: {X_train_cat_imputed.shape}")
print(f"Imputed Validation Numeric Shape: {X_val_num_imputed.shape}")
print(f"Imputed Validation Categorical Shape: {X_val_cat_imputed.shape}")

TECHNICAL NOTE: In order for the SimpleImputer to apply the filling strategies chosen above, Numpy, the mathematical backbone tool of machine learning, had to convert the data type of X from a pandas dataframe (table) to a numpy.ndarray type (numeric format). This is like changing the 'container' of the data, not the data itself. The 'container' is now a sequence of numbers without headings and indexes. But the important thing is that the elements inside the container are still in their original data type, and SimpleImputer has applied the selected filling strategies.

This subtle change happens automatically with almost all sklearn transformers, because sklearn algorithms prefer raw, highly optimized mathematical matrices over the styled dataframes that we simple, non-numeric humans understand.

Below, for example, the `X_train_cat_imputed` piece is examined. (This represent the categorical housing features that will be used for training).

While the data type of `X_train_cat_imputed` as a container has been changed to a numpy array, the elements _inside_ that container are still string objects such as "Pave", "Grvl", "Shed", etc.


In [ ]:
print(f" Data type of X_train_cat_imputed:           {type(X_train_cat_imputed)}")
print(f" Data type inside of X_train_cat_imputed:    {X_train_cat_imputed.dtype}")

**Step 3.4:** Skipped


**Step 3.5:** Apply `OneHotEncoder(handle_unknown='ignore', sparse_output=False)` to categorical text columns to prevent dimension mismatches when checking validation data.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

#Instantiate OneHotEncoder
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Fit and transform the training categorical array
X_train_cat_encoded=ohe.fit_transform(X_train_cat_imputed)

# Transform the validation categorical array without fitting it
X_val_cat_encoded=ohe.transform(X_val_cat_imputed)

# View difference in data shape of imputed, compared to encoded
print(f"Original train cat shape: {X_train_cat_imputed.shape}")
print(f"Encoded train cat array shape: {X_train_cat_encoded.shape}")
print(f"Original val cat array shape: {X_val_cat_imputed.shape}")
print(f"Encoded val cat array shape: {X_val_cat_encoded.shape}")

###Rejoining the data

Now that the categorical data have been converted from text to numbers, they need to be rejoined with the numeric data into a new integrated object.



In [ ]:
X_train_final = np.hstack((X_train_num_imputed, X_train_cat_encoded))
X_val_final = np.hstack((X_val_num_imputed, X_val_cat_encoded))

print(f"Final training matrix shape: {X_train_final.shape}")
print(f"Final validation matrix shape: {X_val_final.shape}")

## 🤖 Phase 4: Baseline Training & Evaluation Framework
*Goal: Build a solid evaluation metric to track your model's true performance.*

### ❓ Core Question: How do I evaluate a regression model?
*   **The Theory:** Classification metrics like accuracy scores do not work when predicting continuous prices. If a house costs $300,000 and your model guesses $295,000, it is technically "wrong," but practically excellent. You must measure real dollar errors using Mean Absolute Error (MAE) for average distance, and Root Mean Squared Error (RMSE) to penalize rare, massive misses.
*   **JIT Reading Anchor:** Read the Scikit-Learn metrics user guide covering `mean_absolute_error` and `root_mean_squared_error`.

### 🛠️ Execution Checklist
- [ ] **Step 4.1:** Write a reusable Python function named `evaluate_regression(y_true, y_pred)` that explicitly prints out calculated MAE, RMSE, and $R^2$ scores.
- [ ] **Step 4.2:** Instantiate a baseline `sklearn.ensemble.RandomForestRegressor`.
- [ ] **Step 4.3:** Fit the regressor on your processed training arrays and output predictions from your validation feature set (`X_val`).
- [ ] **Step 4.4:** Convert your validation predictions back to real dollar limits using `numpy.expm1()` before running them through your evaluation function.

*   *JIT Reading Anchor:* Read the Scikit-Learn documentation metrics module specifically comparing MAE (average dollar error) to RMSE (penalty for large misses).

###4.1 Setting up evaluation metrics
At this point, the training data has been preprocessed into a format that can be passed into a machine learning model.


Before the model can be trained on this data, there need to be some kind of evaluation metrics set up to judge its performance by.

We'll use 3:
* Mean Absolute Error, which is the average dollar amount predictions are off by.
* Root Mean Squared Error, which squares all errors so that large errors are disproportionately penalized.
* R2 Score, which tells how much the *variations* of house prices are being predicted compared to just guessing the mean.

Note: The Root Mean Squared Error is the metric the Kaggle Competition uses to grade submissions for this dataset.

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

def evaluate_regression_predictions(y_true_log, y_pred_log):
    """
    Converts log-transformed prices back to true dollar amounts
    and prints out standard regression metrics. The np.expm function
    will reverse the log1p skewness transformation done earlier in step 2.3
    """

    y_true_dollars = np.expm(y_true_log)
    y_pred_dollars = np.expm(y_pred_log)

    mae = mean_absolute_error(y_true_dollars, y_pred_dollars)
    rmse = root_mean_squared_error(y_true_dollars, y_pred_dollars)
    r2 = r2_score(y_true_dollars, y_pred_dollars)

    print(f"Mean Absolute Error: {mae:,.2f}")
    print(f"Root Mean Squared Error: {mae:,.2f}")
    print(f"R^2 Score (Variance Explained): {r2:.4f}")


Instantiate the Random Forest Regressor model.


In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(random_state=7)

